# 7주차 실습. Single-cell multiome: 염색질 접근성과 RNA

이 파일을 직접 고치지 말고 먼저 본인 폴더로 복사한다. Qoka 대화 입력창에 입력한다.

> course/labs/week7/week7_multiome.ipynb 를 analysis/week7/ 폴더로 복사해서 열어줘.

열리면 오른쪽 위 커널을 **Qoka Run Environment**로 고른다.

## 이 노트북을 쓰는 방법

- SnapATAC2 튜토리얼 "Standard pipeline"의 단계와 multiome 비교를 셀 하나씩 나눔. 코드는 미리 적혀 있음
- 셀마다: **프롬프트**를 읽는다 → 코드에서 프롬프트의 말이 된 줄을 찾는다 → **예상**을 적는다 → 실행하고 비교한다. 결과 설명은 강의노트
- 셀은 위에서부터 순서대로 실행

| 실습 | 셀 | 내용 | 강의노트 |
|---|---|---|---|
| 1 | 1–6 | fragment 파일, QC (fragment 크기, TSS enrichment) | 7.4–7.5 |
| 2 | 7–12 | tile 행렬, spectral, clustering, RNA로 이름 붙이기, gene activity | 7.6–7.9 |
| 3 | 13–17 | marker peak, peak–유전자 상관과 비교 기준 | 7.10–7.11 |

## 준비 (수업 전에)

패키지를 설치하고 자료(약 590 MB)를 받는다. **수업 전에 이 세 셀까지 실행해 둔다.** "restart the kernel"이 나오면 커널 재시작 후 처음부터.

In [ ]:
# SnapATAC2가 peak calling에 쓰는 macs3는 설치할 때 컴파일이 필요해서 빼고 설치 (이 노트북은 macs3를 쓰지 않음)
%pip install -q snapatac2==2.10.0 --no-deps
%pip install -q scanpy==1.11.4 leidenalg igraph matplotlib "pandas<3" "numpy<3" "polars<2" "pooch<2" "pyfaidx<0.9" rustworkx typeguard natsort multiprocess "plotly<7" kaleido "scverse-misc[datasets]<0.2" pyarrow

In [ ]:
import gzip
import itertools
import shutil
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import snapatac2 as snap
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings("ignore", category=FutureWarning)   # 라이브러리 내부 경고 (5주차 5.5절)

# data/ 폴더가 나올 때까지 위로 올라가며 프로젝트 폴더를 찾음 (3주차와 같음)
root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
data_dir = root / "data" / "week7"
out_dir = root / "results" / "week7"      # 결과를 저장할 곳
data_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)
sc.settings.verbosity = 0
print(root)

In [ ]:
# 10x Genomics 공개 자료 PBMC 3k multiome (cellranger-arc 2.0). 같은 세포 2,711개에서 RNA와 ATAC
URL = "https://cf.10xgenomics.com/samples/cell-arc/2.0.0/pbmc_granulocyte_sorted_3k/"
PREFIX = "pbmc_granulocyte_sorted_3k_"
FILES = {"atac_fragments.tsv.gz": 467.6,        # fragment 하나 = 한 줄
         "atac_fragments.tsv.gz.tbi": 0.7,      # 위 파일의 index (IGV에서 씀)
         "filtered_feature_bc_matrix.h5": 38.8, # 세포 × (유전자 + peak)
         "per_barcode_metrics.csv": 77.7,       # 바코드마다 QC 값
         "atac_peak_annotation.tsv": 5.5}       # peak마다 가까운 유전자

for name, mb in FILES.items():
    f = data_dir / (PREFIX + name)
    if not f.exists():
        tmp = f.with_name(f.name + ".part")             # 받다가 끊기면 .part로 남음
        req = urllib.request.Request(URL + PREFIX + name, headers={"User-Agent": "Mozilla/5.0"})   # 기본값(Python-urllib)은 403
        with urllib.request.urlopen(req) as r, open(tmp, "wb") as out:
            shutil.copyfileobj(r, out, 1 << 20)
        tmp.rename(f)
    print(f"{f.name:58s} {f.stat().st_size / 1e6:6.1f} MB  (10x 목록 {mb} MB)")

_ = snap.genome.hg38.annotation, snap.genome.hg19.annotation   # TSS 주석 (셀 5). 처음 한 번 받음

---
# 실습 1. Fragment와 QC

## 셀 1. Fragment 파일

**프롬프트**: > fragment 파일의 앞부분을 보여줘.

**예상**: 한 줄에 열이 5개. 각각 무엇일까? (4주차 BED: 염색체, 시작, 끝)

> (실행하기 전에 여기에 적는다)

In [ ]:
frag_file = data_dir / (PREFIX + "atac_fragments.tsv.gz")
with gzip.open(frag_file, "rt") as f:
    lines = list(itertools.islice(f, 100))
comments = [l for l in lines if l.startswith("#")]      # 파일 앞의 설명 줄
print(f"설명 줄 {len(comments)}개. 예: {comments[1].strip()}")
print("".join([l for l in lines if not l.startswith("#")][:5]))

## 셀 2. 바코드와 세포

**프롬프트**: > 바코드 중 세포로 판정된 것이 몇 개인지 보여줘.

**예상**: 바코드는 몇 개, 그중 세포는 몇 개일까? 세포가 아닌 바코드에는 fragment가 몇 개씩 있을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
metrics = pd.read_csv(data_dir / (PREFIX + "per_barcode_metrics.csv"))
cells = metrics[metrics.is_cell == 1]                   # cellranger-arc가 세포로 판정한 바코드
print(f"바코드 {len(metrics):,}개 중 세포 {len(cells):,}개")
print(f"세포가 아닌 바코드의 fragment 수 중앙값: {metrics[metrics.is_cell == 0].atac_fragments.median():.0f}")
print(f"전체 fragment 중 세포에 속한 비율: {cells.atac_fragments.sum() / metrics.atac_fragments.sum():.1%}")
print(f"세포의 FRiP (peak 안 fragment 비율) 중앙값: {(cells.atac_peak_region_fragments / cells.atac_fragments).median():.2f}")

## 셀 3. 세포의 fragment 읽기

**프롬프트**: > 세포 바코드의 fragment만 읽어줘.

**예상**: `frac_mito`(미토콘드리아 fragment 비율)는 얼마일까? (6주차 Kang 자료, 강의노트 7.5절)

> (실행하기 전에 여기에 적는다)

In [ ]:
atac = snap.pp.import_fragments(frag_file, chrom_sizes=snap.genome.hg38,   # 자료의 genome: GRCh38 = hg38
                                whitelist=cells.barcode.tolist(),        # 세포 바코드만
                                sorted_by_barcode=False, min_num_fragments=0)
print(atac)                                   # 행 = 세포. 열(구간)은 셀 7에서 만듦
atac.obs[["n_fragment", "frac_dup", "frac_mito"]].describe().round(3)

## 셀 4. Fragment 길이

**프롬프트**: > fragment 길이 분포를 그려줘.

**예상**: 봉우리는 몇 개, 어디에 있을까? (강의노트 7.3절)

> (실행하기 전에 여기에 적는다)

In [ ]:
snap.metrics.frag_size_distr(atac, add_key="frag_size")
fs = np.asarray(atac.uns["frag_size"], dtype=float)
fs = fs / fs.sum()                                     # 길이마다 fragment 비율

plt.figure(figsize=(6, 3))
plt.plot(np.arange(len(fs)), fs * 100)
plt.xlim(0, 800); plt.xlabel("fragment length (bp)"); plt.ylabel("% of fragments")
plt.show()
for lo, hi in [(30, 120), (130, 260), (280, 460)]:
    print(f"{lo}–{hi} bp에서 가장 많은 길이: {lo + fs[lo:hi].argmax()} bp")
print(f"100 bp 미만: {fs[:100].sum():.0%}")

## 셀 5. TSS enrichment

**프롬프트**: > TSS enrichment를 계산해줘.

**예상**: TSS 주석을 예전 genome(hg19)으로 쓰면 값은? 오류가 날까? (강의노트 7.5절)

> (실행하기 전에 여기에 적는다)

In [ ]:
for build in ["hg19", "hg38"]:                         # 자료는 hg38. 마지막에 hg38 값이 남음
    snap.metrics.tsse(atac, getattr(snap.genome, build))
    print(f"{build}: TSSe 중앙값 {atac.obs.tsse.median():.1f}, 5 이상인 세포 {(atac.obs.tsse >= 5).sum()}개 / {atac.n_obs}")

plt.figure(figsize=(4.5, 3.5))
plt.scatter(atac.obs.n_fragment, atac.obs.tsse, s=3, alpha=0.5)
plt.xscale("log"); plt.xlabel("fragments per cell"); plt.ylabel("TSS enrichment (hg38)")
plt.axvline(1000, ls="--", c="k"); plt.axhline(5, ls="--", c="k")
plt.show()

## 셀 6. 세포 거르기

**프롬프트**: > fragment 수와 TSS enrichment 기준으로 세포를 걸러줘.

**예상**: 세포는 몇 개 남을까? (SnapATAC2 튜토리얼 기준: fragment 1,000–100,000, TSSe 5)

> (실행하기 전에 여기에 적는다)

In [ ]:
o = atac.obs
print(f"fragment < 1,000: {(o.n_fragment < 1000).sum()}, ≥ 100,000: {(o.n_fragment >= 100000).sum()}, TSSe < 5: {(o.tsse < 5).sum()}")
snap.pp.filter_cells(atac, min_counts=1000, max_counts=100000, min_tsse=5)
atac.n_obs

---
# 실습 2. 행렬, clustering, 이름 붙이기

## 셀 7. Tile 행렬

**프롬프트**: > 500 bp 구간마다 fragment를 세어 행렬을 만들어줘.

**예상**: 0인 칸은 몇 %일까? 0이 아닌 값은 주로 몇일까? (6주차 RNA: 97.4%)

> (실행하기 전에 여기에 적는다)

In [ ]:
snap.pp.add_tile_matrix(atac, bin_size=500)             # 유전체 전체를 500 bp 구간(tile)으로 나눔
X = atac.X
print(f"세포 × 구간: {atac.shape}")
print(f"0의 비율: {1 - X.nnz / (X.shape[0] * X.shape[1]):.2%}")
print(f"0이 아닌 값: 1이 {np.mean(X.data == 1):.0%}, 2가 {np.mean(X.data == 2):.0%}, 최대 {X.data.max():.0f}")

## 셀 8. Doublet과 spectral embedding

**프롬프트**: > doublet을 거르고 spectral embedding을 해줘.

**예상**: 1번 성분은 무엇과 상관이 클까? (6주차 PC1: 단핵구 vs 림프구. 강의노트 7.7절)

> (실행하기 전에 여기에 적는다)

In [ ]:
snap.pp.select_features(atac, n_features=50000)        # 자주 열리는 구간 50,000개
snap.pp.scrublet(atac, random_state=0)                  # doublet 점수 (가짜 doublet과 비교)
snap.pp.filter_doublets(atac)
print("doublet 제거 후:", atac.n_obs)

snap.tl.spectral(atac)                                  # PCA와 같은 역할. 결과는 obsm["X_spectral"]
depth = np.log10(atac.obs.n_fragment)
for i in range(5):
    print(f"{i + 1}번 성분과 log(fragment 수)의 상관: {np.corrcoef(atac.obsm['X_spectral'][:, i], depth)[0, 1]:+.2f}")

## 셀 9. 1번 성분을 넣을 때와 뺄 때

**프롬프트**: > 1번 성분을 넣을 때와 뺄 때 Leiden clustering을 비교해줘.

**예상**: cluster마다 fragment 수 중앙값을 보면? 유난히 적은 cluster가 생길까?

> (실행하기 전에 여기에 적는다)

In [ ]:
n = atac.obsm["X_spectral"].shape[1]
for key, dims in [("with_1", None), ("without_1", list(range(1, n)))]:   # None = 모든 성분
    snap.pp.knn(atac, use_dims=dims, n_neighbors=15)
    snap.tl.leiden(atac, resolution=1.0, random_state=0, key_added=key)
    med = atac.obs.groupby(key, observed=True).n_fragment.median()
    print(f"{key:9s}: cluster {len(med)}개, cluster별 fragment 중앙값 {med.min():,.0f}–{med.max():,.0f}")

print("1번 포함, fragment가 적은 cluster:")
print(atac.obs.groupby("with_1", observed=True).n_fragment.agg(["size", "median"]).sort_values("median").head(3))
snap.tl.umap(atac, use_dims=list(range(1, n)), random_state=0)       # 그림도 1번을 빼고
sc.pl.embedding(atac, "X_umap", color=["with_1", "without_1"], legend_loc="on data")

## 셀 10. 같은 세포의 RNA

**프롬프트**: > 같은 세포의 RNA로 cell type 이름을 붙여줘.

**예상**: 6주차 미토콘드리아 기준(5%)을 그대로 쓰면 세포가 몇 개 남을까? (강의노트 7.8절)

> (실행하기 전에 여기에 적는다)

In [ ]:
m = sc.read_10x_h5(data_dir / (PREFIX + "filtered_feature_bc_matrix.h5"), gex_only=False)
print(m.var.feature_types.value_counts().to_dict(), "| genome:", m.var.genome.unique())

rna = m[:, m.var.feature_types == "Gene Expression"].copy()
rna.var_names_make_unique()
rna.var["mt"] = rna.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
print(f"미토콘드리아 비율 중앙값 {rna.obs.pct_counts_mt.median():.1f}%, 5% 미만인 세포 {(rna.obs.pct_counts_mt < 5).sum()}개 / {rna.n_obs}")

# 6주차 단계 (기준은 이 자료에 맞춤: 미토콘드리아 20%, 교재 5.2절 값)
rna = rna[(rna.obs.n_genes_by_counts >= 200) & (rna.obs.n_genes_by_counts < 5000) & (rna.obs.pct_counts_mt < 20)].copy()
sc.pp.filter_genes(rna, min_cells=3)
rna.layers["counts"] = rna.X.copy()                     # 셀 12, 14에서 씀
sc.pp.normalize_total(rna, target_sum=1e4)
sc.pp.log1p(rna)
sc.pp.highly_variable_genes(rna, n_top_genes=2000)
sc.tl.pca(rna, n_comps=30, mask_var="highly_variable", random_state=0)
sc.pp.neighbors(rna, n_neighbors=15, n_pcs=30, random_state=0)
sc.tl.leiden(rna, resolution=1.0, random_state=0, flavor="igraph", n_iterations=2, directed=False)

markers = {"CD4 naive": ["IL7R", "CCR7", "LEF1"], "CD8": ["CD8A", "GZMK"], "NK": ["NKG7", "GNLY"],
           "B": ["MS4A1", "CD79A"], "CD14 Mono": ["CD14", "LYZ"], "FCGR3A Mono": ["FCGR3A", "MS4A7"],
           "cDC": ["FCER1A", "CLEC10A"], "pDC": ["LILRA4", "IL3RA"]}
sc.pl.dotplot(rna, markers, groupby="leiden")

# cluster 번호는 실행 환경마다 다름 (리허설: 서버 12개, Mac 13개) → 번호가 아니라 marker 평균으로 이름
rules = {  # cell type: (높아야 하는 marker, 낮아야 하는 marker)
    "CD4 naive": (["IL7R", "CCR7", "LEF1"], ["CD8A", "NKG7"]),
    "CD4 memory": (["IL7R"], ["CCR7", "CD8A", "NKG7"]),
    "CD8 naive": (["CD8A", "CCR7", "LEF1"], ["GZMK", "NKG7"]),
    "CD8 memory": (["CD8A", "GZMK", "NKG7"], ["CCR7"]),
    "NK": (["NKG7", "GNLY"], ["CD3E"]),
    "B": (["MS4A1", "CD79A"], []),
    "CD14 Mono": (["CD14"], ["FCGR3A", "FCER1A"]),
    "FCGR3A Mono": (["FCGR3A", "MS4A7"], ["CD14", "NKG7"]),
    "cDC": (["FCER1A", "CLEC10A"], []),
    "pDC": (["LILRA4", "IL3RA"], []),
}
genes = sorted({g for up, down in rules.values() for g in up + down} | {"LYZ"})
avg = sc.get.obs_df(rna, keys=genes + ["leiden"]).groupby("leiden", observed=True).mean()   # cluster × marker 평균
score = pd.DataFrame({t: avg[up].mean(axis=1) - (avg[down].mean(axis=1) if down else 0) for t, (up, down) in rules.items()})
top2 = np.sort(score.values, axis=1)[:, -2:]
names = pd.DataFrame({"세포": rna.obs.leiden.value_counts().reindex(score.index).values, "이름": score.idxmax(axis=1),
                      "점수": top2[:, 1].round(2), "2위와 차이": (top2[:, 1] - top2[:, 0]).round(2)})
names.loc[(avg.LYZ > 1) & (avg.CD3E > 0.25), "이름"] = "mixed"     # 단핵구와 T marker가 함께 높음 → doublet 의심
print(names)                                                         # 2위와 차이가 작은 cluster는 dotplot으로 다시 확인

rna.obs["cell_type"] = rna.obs["leiden"].map(names["이름"]).astype(str)
rna = rna[rna.obs.cell_type != "mixed"].copy()
rna.obs["cell_type"].value_counts()

## 셀 11. ATAC cluster에 RNA 이름 옮기기

**프롬프트**: > RNA의 cell type 이름을 ATAC 세포에 옮기고 ATAC cluster와 비교해줘.

**예상**: ATAC cluster 하나에 cell type이 하나씩 들어갈까? 잘 나뉘지 않는 cell type은?

> (실행하기 전에 여기에 적는다)

In [ ]:
both = atac.obs_names.intersection(rna.obs_names)        # 같은 바코드 = 같은 세포
atac = atac[both].copy()
rna = rna[both].copy()
atac.obs["cell_type"] = rna.obs["cell_type"].values
print(f"ATAC과 RNA를 모두 통과한 세포: {len(both)}개")
print(f"ATAC cluster와 RNA cell type의 일치도 (ARI): {adjusted_rand_score(atac.obs.without_1, atac.obs.cell_type):.2f}")
sc.pl.embedding(atac, "X_umap", color="cell_type")
pd.crosstab(atac.obs["without_1"], atac.obs["cell_type"])

## 셀 12. Gene activity

**프롬프트**: > 유전자마다 열린 정도(gene activity)를 계산해서 RNA와 비교해줘.

**예상**: marker 유전자는 열린 곳과 발현된 곳이 같을까? 모든 유전자에서도 그럴까? (강의노트 7.9절)

> (실행하기 전에 여기에 적는다)

In [ ]:
ga = snap.pp.make_gene_matrix(atac, snap.genome.hg38)   # 유전자 몸통 + promoter의 fragment 수
genes = ga.var_names.intersection(rna.var_names)

def by_type(X, var_names, labels):
    """cell type마다 세포의 count를 더해 CPM의 log (셀 15의 metacell과 같은 방식)"""
    rows = {t: np.asarray(X[(labels == t).values].sum(axis=0)).ravel() for t in sorted(labels.unique())}
    df = pd.DataFrame(rows, index=var_names)
    return np.log1p(df / df.sum() * 1e6)

R = by_type(rna.layers["counts"], rna.var_names, rna.obs.cell_type).loc[genes]
A = by_type(ga.X, ga.var_names, atac.obs.cell_type).loc[genes]
ok = (R.max(axis=1) > np.log1p(20)) & (A.max(axis=1) > np.log1p(20))   # 두 쪽 모두 어딘가에서는 검출
r = pd.Series([np.corrcoef(R.loc[g], A.loc[g])[0, 1] for g in genes[ok]], index=genes[ok])
print(f"유전자 {ok.sum()}개, cell type 사이 상관(RNA vs gene activity) 중앙값 {r.median():.2f}, 0.5 초과 {np.mean(r > 0.5):.0%}, 음수 {np.mean(r < 0):.0%}")

show = ["CD14", "MS4A1", "PAX5", "CD3E", "CD8A", "NKG7", "FCGR3A", "ACTB"]
print(pd.DataFrame({"r": r[show].round(2), "RNA 최고": R.loc[show].idxmax(axis=1), "activity 최고": A.loc[show].idxmax(axis=1)}))

---
# 실습 3. Peak와 유전자

## 셀 13. Peak 행렬과 marker peak

**프롬프트**: > cell type마다 특히 열린 peak를 찾고 가까운 유전자를 보여줘.

**예상**: 10x peak 행렬의 값은 주로 몇일까? (셀 7의 tile 행렬과 비교) B cell marker peak 근처에는 어떤 유전자가 있을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
pk = m[both, m.var.feature_types == "Peaks"].copy()      # cellranger-arc가 부른 peak
v = pk.X.data
print(f"peak {pk.n_vars:,}개. 0이 아닌 값: 1이 {np.mean(v == 1):.0%}, 2가 {np.mean(v == 2):.0%}, 3 이상 {np.mean(v > 2):.0%}")
pk.X = (pk.X > 0).astype(np.float32)                    # 열렸다(1) / 아니다(0)
pk.obs["cell_type"] = atac.obs["cell_type"].values

marker_peaks = snap.tl.marker_regions(pk, groupby="cell_type", pvalue=0.01)
print({t: len(p) for t, p in marker_peaks.items()})

ann = pd.read_csv(data_dir / (PREFIX + "atac_peak_annotation.tsv"), sep="\t")      # peak → 가까운 유전자
ann["peak"] = ann.chrom + ":" + ann.start.astype(str) + "-" + ann.end.astype(str)
for t in ["B", "CD14 Mono", "NK"]:
    near = ann[ann.peak.isin(marker_peaks[t])].gene.value_counts()
    print(f"{t}: marker peak가 많은 유전자 {near.head(8).to_dict()}")

## 셀 14. Metacell

**프롬프트**: > RNA가 비슷한 세포를 60개 묶음으로 나누고 묶음마다 RNA와 peak를 합쳐줘.

**예상**: 세포 하나 단위로 상관을 구하지 않는 이유는? (셀 12: 세포 대부분이 0)

> (실행하기 전에 여기에 적는다)

In [ ]:
mc = KMeans(60, random_state=0, n_init=4).fit_predict(rna.obsm["X_pca"])   # RNA PC 30개로 60묶음
print(f"묶음당 세포 {np.bincount(mc).min()}–{np.bincount(mc).max()}개")

def by_metacell(X):
    """묶음마다 count를 더해 CPM의 log. 행 = 묶음 60개"""
    S = np.vstack([np.asarray(X[mc == c].sum(axis=0)).ravel() for c in range(60)])
    return np.log1p(S / S.sum(axis=1, keepdims=True) * 1e6)

RM = by_metacell(rna.layers["counts"])                 # 묶음 × 유전자
PM = by_metacell(pk.X)                                 # 묶음 × peak
mc_type = np.array([atac.obs.cell_type[mc == c].value_counts().index[0] for c in range(60)])
print(pd.Series(mc_type).value_counts().to_dict())

## 셀 15. 유전자 하나의 주변 peak

**프롬프트**: > LYZ 주변 100 kb 안의 peak와 LYZ 발현의 상관을 보여줘.

**예상**: 상관이 0.5 넘는 peak는 몇 개일까? 이 peak들이 LYZ를 조절한다고 할 수 있을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
loc = pk.var_names.str.split(r"[:-]", regex=True)      # 예: chr1:9768-10660 → [chr1, 9768, 10660]
chrom = np.array([p[0] for p in loc])
mid = np.array([(int(p[1]) + int(p[2])) // 2 for p in loc])   # peak 가운데 위치
gene_pos = m.var.loc[m.var.feature_types == "Gene Expression", "interval"]
gene_pos = gene_pos[~gene_pos.index.duplicated()]      # 유전자 위치 (예: chr12:69348341-69354234)

def corr(x, Y):
    """x(묶음 60개)와 Y의 각 열의 Pearson 상관"""
    x = x - x.mean(); Y = Y - Y.mean(axis=0)
    return (Y.T @ x) / (np.linalg.norm(Y, axis=0) * np.linalg.norm(x) + 1e-12)

def cis_peaks(gene, flank=100_000):
    ch, se = gene_pos[gene].split(":"); s, e = map(int, se.split("-"))
    return np.where((chrom == ch) & (mid > s - flank) & (mid < e + flank))[0]

g = "LYZ"
cis = cis_peaks(g)
r_cis = corr(RM[:, rna.var_names.get_loc(g)], PM[:, cis])
print(f"{g} 주변 peak {len(cis)}개, r > 0.5: {(r_cis > 0.5).sum()}개")
pd.DataFrame({"peak": pk.var_names[cis], "r": r_cis.round(2)}).sort_values("r", ascending=False).head(5)

## 셀 16. 다른 염색체의 peak와 비교

**프롬프트**: > 같은 계산을 다른 염색체의 peak로 해서 비교해줘.

**예상**: 다른 염색체의 peak는 이 유전자를 조절할 수 없다. 상관이 0.5 넘는 비율은 주변 peak보다 얼마나 작을까? (강의노트 7.11절)

> (실행하기 전에 여기에 적는다)

In [ ]:
rng = np.random.default_rng(0)
level = np.digitize(PM.mean(axis=0), np.quantile(PM.mean(axis=0), np.linspace(0.1, 0.9, 9)))   # 열린 정도 10단계

def trans_peaks(cis):
    """cis peak마다 다른 염색체에서 열린 정도가 같은 peak 하나를 무작위로"""
    return np.array([rng.choice(np.where((chrom != chrom[j]) & (level == level[j]))[0]) for j in cis])

rows = []
for gene in rna.var_names[rna.var.highly_variable]:    # HVG 2,000개
    if gene not in gene_pos.index or ":" not in str(gene_pos[gene]):
        continue
    cis = cis_peaks(gene)
    x = RM[:, rna.var_names.get_loc(gene)]
    if len(cis) == 0 or x.std() == 0:
        continue
    for kind, idx in [("cis", cis), ("trans", trans_peaks(cis))]:
        rows += [(gene, kind, v) for v in corr(x, PM[:, idx])]
pg = pd.DataFrame(rows, columns=["gene", "kind", "r"])
print(f"유전자 {pg.gene.nunique()}개, 쌍 {(pg.kind == 'cis').sum():,}개씩")
print(pg.groupby("kind").r.agg(n="size", r_over_05=lambda s: np.mean(s > 0.5), mean_r="mean").round(3))
print(f"LYZ 최대 r: cis {pg[(pg.gene == 'LYZ') & (pg.kind == 'cis')].r.max():.2f}, trans {pg[(pg.gene == 'LYZ') & (pg.kind == 'trans')].r.max():.2f}")

## 셀 17. 같은 cell type 안에서

**프롬프트**: > cell type 차이를 뺀 뒤에 다시 비교해줘.

**예상**: cell type 평균을 빼면 cis와 trans의 r > 0.5 비율은 어떻게 될까?

> (실행하기 전에 여기에 적는다)

In [ ]:
def remove_type_mean(Y):
    """묶음마다 그 cell type 묶음들의 평균을 뺌 → cell type 사이 차이 제거"""
    Y = Y.copy()
    for t in np.unique(mc_type):
        Y[mc_type == t] -= Y[mc_type == t].mean(axis=0)
    return Y

RMw, PMw = remove_type_mean(RM), remove_type_mean(PM)
rng = np.random.default_rng(0)                          # 셀 16과 같은 trans peak
rows = []
for gene in pg.gene.unique():
    cis = cis_peaks(gene)
    x = RMw[:, rna.var_names.get_loc(gene)]
    for kind, idx in [("cis", cis), ("trans", trans_peaks(cis))]:
        rows += [(gene, kind, v) for v in corr(x, PMw[:, idx])]
pw = pd.DataFrame(rows, columns=["gene", "kind", "r"])
print(pw.groupby("kind").r.agg(r_over_05=lambda s: np.mean(s > 0.5), mean_r="mean").round(3))

pg.to_csv(out_dir / "peak_gene_cis_trans.csv", index=False)
atac.obs.to_csv(out_dir / "atac_cells.csv")
slim = sc.AnnData(obs=atac.obs, obsm={"fragment_paired": atac.obsm["fragment_paired"], "X_umap": atac.obsm["X_umap"]},
                  uns={"reference_sequences": atac.uns["reference_sequences"]})
slim.write(out_dir / "pbmc3k_atac_fragments.h5ad", compression="gzip")   # 실습 3.5에서 씀 (fragment + cell type)

---
# 다음

- **실습 3.5**: Qoka 대화 입력창에서 프롬프트 레파토리 (강의노트 7.12절). 에이전트가 쓰는 코드는 `analysis/week7/`
- **실습 4**: 커널 재시작 후 전체 실행 (Restart & Run All), 커밋 (강의노트 7.13절)